In [ ]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 22.8 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which 

In [ ]:
from datasets import load_dataset
raw_dataset=load_dataset('conll2003')

README.md:   0%|          | 0.00/12.3k [00:00<?, ?B/s]

conll2003.py:   0%|          | 0.00/9.57k [00:00<?, ?B/s]

The repository for conll2003 contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/conll2003.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3453 [00:00<?, ? examples/s]

In [ ]:
raw_dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

In [ ]:
###first remove and rename columns
raw_dataset=raw_dataset.remove_columns(['id','chunk_tags','ner_tags'])
raw_dataset=raw_dataset.rename_column('tokens','words')
raw_dataset=raw_dataset.rename_column('pos_tags','labels')
raw_dataset

DatasetDict({
    train: Dataset({
        features: ['words', 'labels'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['words', 'labels'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['words', 'labels'],
        num_rows: 3453
    })
})

In [ ]:
print(raw_dataset['train']['words'][0])
raw_dataset['train']['labels'][0]

['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.']


[22, 42, 16, 21, 35, 37, 16, 21, 7]

In [ ]:
pos_features=raw_dataset['train'].features['labels']
labels_name=pos_features.feature.names

In [ ]:
labels_name

['"',
 "''",
 '#',
 '$',
 '(',
 ')',
 ',',
 '.',
 ':',
 '``',
 'CC',
 'CD',
 'DT',
 'EX',
 'FW',
 'IN',
 'JJ',
 'JJR',
 'JJS',
 'LS',
 'MD',
 'NN',
 'NNP',
 'NNPS',
 'NNS',
 'NN|SYM',
 'PDT',
 'POS',
 'PRP',
 'PRP$',
 'RB',
 'RBR',
 'RBS',
 'RP',
 'SYM',
 'TO',
 'UH',
 'VB',
 'VBD',
 'VBG',
 'VBN',
 'VBP',
 'VBZ',
 'WDT',
 'WP',
 'WP$',
 'WRB']

In [ ]:
words=raw_dataset['train']['words'][0]
labels=raw_dataset['train']['labels'][0]
line1=''
line2=''
for word , label in zip(words,labels):
  full_label=labels_name[label]
  max_lenght=max(len(word),len(full_label))
  line1+=word+" "*(max_lenght-len(word)+1)
  line2+=full_label+' '*(max_lenght-len(full_label)+1)
print(line1)
print(line2)

EU  rejects German call to boycott British lamb . 
NNP VBZ     JJ     NN   TO VB      JJ      NN   . 


In [ ]:
###let's define tokenizer
from transformers import AutoTokenizer
model_id='bert-base-cased'
tokenizer=AutoTokenizer.from_pretrained(model_id)

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

In [ ]:
raw_dataset['train']['words'][0]

['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.']

In [ ]:
###becuse they are split
tokenize=tokenizer(raw_dataset['train']['words'][0],truncation=True,is_split_into_words=True)
tokenize.tokens()

['[CLS]',
 'EU',
 'rejects',
 'German',
 'call',
 'to',
 'boycott',
 'British',
 'la',
 '##mb',
 '.',
 '[SEP]']

In [ ]:
tokenize.word_ids()

[None, 0, 1, 2, 3, 4, 5, 6, 7, 7, 8, None]

In [ ]:
def align_labels_with_tokens(labels,word_ids):
  new_labels=[]
  current_word=None

  for word_id in word_ids:
    if word_ids!=current_word:
      current_word=word_id
      label=-100 if word_id is None else labels[word_id]
      new_labels.append(label)
    elif word_ids == None:
      new_labels.append(-100)
    else:
      label=labels[word_id]
      new_labels.append(label)
  return new_labels

In [ ]:
labels=raw_dataset['train']['labels'][0]
word_id=tokenize.word_ids()
print(labels)
print(align_labels_with_tokens(labels,word_id))

[22, 42, 16, 21, 35, 37, 16, 21, 7]
[-100, 22, 42, 16, 21, 35, 37, 16, 21, 21, 7, -100]


In [ ]:
def tokenize_and_align_labels(examples):
  tokenized_input=tokenizer(examples['words'],
                            truncation=True,
                            is_split_into_words=True)
  all_labels=examples['labels']
  new_labels=[]
  for i,label in enumerate(all_labels):
    word_ids=tokenized_input.word_ids(i)
    new_labels.append(align_labels_with_tokens(label,word_ids))
  tokenized_input["labels"] = new_labels
  return tokenized_input

In [ ]:
tokenize_dataset=raw_dataset.map(tokenize_and_align_labels,
                                 batched=True,
                                  remove_columns=raw_dataset["train"].column_names)

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

In [ ]:
tokenize_dataset

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3453
    })
})

In [ ]:
from transformers import DataCollatorForTokenClassification
data_collator=DataCollatorForTokenClassification(tokenizer=tokenizer)

In [ ]:
!pip install seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=8c6ecd451f1a466b3e6e615f36b5de1730c38a67118c292bc6c0177f4397bc3a
  Stored in directory: /root/.cache/pip/wheels/bc/92/f0/243288f899c2eacdfa8c5f9aede4c71a9bad0ee26a01dc5ead
Successfully built seqeval


In [ ]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 8.7 MB/s eta 0:00:00


In [ ]:
import evaluate
metric=evaluate.load('seqeval')

In [ ]:
import numpy as np
def compute_metrics(eval_preds):
  logits, labels=eval_preds
  predictions=np.argmax(logits,axis=-1)
  true_labels=[[labels_name[l] for l in label_name if l!=-100]for label_name in labels]
  true_predictions=[
      [labels_name[p] for (p,l) in zip(prediction,label) if l!=-100] for (prediction,label) in zip(predictions,labels)
  ]
  all_metrics = metric.compute(predictions=true_predictions, references=true_labels)
  return {
        "precision": all_metrics["overall_precision"],
        "recall": all_metrics["overall_recall"],
        "f1": all_metrics["overall_f1"],
        "accuracy": all_metrics["overall_accuracy"],
    }

In [ ]:
id2label={i:label for i,label in enumerate(labels_name)}
label2id={label:i for i,label in id2label.items()}

In [ ]:
#define our model
from transformers import AutoModelForTokenClassification
model=AutoModelForTokenClassification.from_pretrained(model_id,
                                                      id2label=id2label,
                                                      label2id=label2id)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
model.config.num_labels

47

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

Token has not been saved to git credential helper.


In [ ]:
from transformers import TrainingArguments
args=TrainingArguments(
    "bert-finetuned-pos",

    save_strategy="epoch",
    learning_rate=2e-5,
    num_train_epochs=3,
    weight_decay=0.01,
    push_to_hub=True,
    )

In [ ]:
from transformers import Trainer
trainer =Trainer(
    model=model,
    args=args,
    train_dataset=tokenize_dataset['train'],
    eval_dataset=tokenize_dataset['validation'],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
)
trainer.train()

Step,Training Loss
500,0.153400
1000,0.122000
1500,0.112000
2000,0.096400
2500,0.074400
3000,0.076500
3500,0.079200
4000,0.051900
4500,0.059400
5000,0.057200


TrainOutput(global_step=5268, training_loss=0.08724395379752214, metrics={'train_runtime': 566.4878, 'train_samples_per_second': 74.358, 'train_steps_per_second': 9.299, 'total_flos': 921087900564942.0, 'train_loss': 0.08724395379752214, 'epoch': 3.0})

In [ ]:
trainer.push_to_hub(commit_message="Training complete")

CommitInfo(commit_url='https://huggingface.co/Alireza0017/bert-finetuned-pos/commit/c883967d29d9ffd8d1beaf0460e53d481d3d5b75', commit_message='Training complete', commit_description='', oid='c883967d29d9ffd8d1beaf0460e53d481d3d5b75', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Alireza0017/bert-finetuned-pos', endpoint='https://huggingface.co', repo_type='model', repo_id='Alireza0017/bert-finetuned-pos'), pr_revision=None, pr_num=None)

In [ ]:
from transformers import pipeline

# Replace this with your own checkpoint
model_checkpoint = "Alireza0017/bert-finetuned-pos"
token_classifier = pipeline(
    "token-classification", model=model_checkpoint, aggregation_strategy="simple"
)
token_classifier("My name is Sylvain and I work at Hugging Face in Brooklyn.")

config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/431M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.22k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/669k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Device set to use cuda:0


[{'entity_group': 'PRP$',
  'score': np.float32(0.99832433),
  'word': 'My',
  'start': 0,
  'end': 2},
 {'entity_group': 'NN',
  'score': np.float32(0.99929976),
  'word': 'name',
  'start': 3,
  'end': 7},
 {'entity_group': 'VBZ',
  'score': np.float32(0.9989225),
  'word': 'is',
  'start': 8,
  'end': 10},
 {'entity_group': 'NNP',
  'score': np.float32(0.9995262),
  'word': 'Sylvain',
  'start': 11,
  'end': 18},
 {'entity_group': 'CC',
  'score': np.float32(0.99983),
  'word': 'and',
  'start': 19,
  'end': 22},
 {'entity_group': 'PRP',
  'score': np.float32(0.99920964),
  'word': 'I',
  'start': 23,
  'end': 24},
 {'entity_group': 'VBP',
  'score': np.float32(0.9940274),
  'word': 'work',
  'start': 25,
  'end': 29},
 {'entity_group': 'IN',
  'score': np.float32(0.99970883),
  'word': 'at',
  'start': 30,
  'end': 32},
 {'entity_group': 'VBG',
  'score': np.float32(0.83306557),
  'word': 'Hugging',
  'start': 33,
  'end': 40},
 {'entity_group': 'NNP',
  'score': np.float32(0.49067